In [2]:
import os
import json
import time
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, AdaBoostClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
warnings.filterwarnings('ignore')

dataset_number = 2
split_ratios = [0.7, 0.8, 0.9]
use_tuned = True
top_mode = 'global'
res_dir = '../Results/'

base_names = ['Random Forest', 'Logistic Regression', 'Decision Tree', 'XGBoost', 'CatBoost', 'AdaBoost', 'LightGBM', 'Support Vector Machine', 'K-Nearest Neighbors', 'Extra Trees', 'Gradient Boosting', 'Naive Bayes']

best_params = json.load(open(res_dir + 'best_params.json')) if os.path.exists(res_dir + 'best_params.json') else {}
top_info = json.load(open(res_dir + 'top_classifiers.json')) if os.path.exists(res_dir + 'top_classifiers.json') else {}

In [3]:
def make_model(name, params={}):
    models = {
        'Random Forest': RandomForestClassifier(random_state=42),
        'Logistic Regression': LogisticRegression(max_iter=3000),
        'Decision Tree': DecisionTreeClassifier(random_state=42),
        'XGBoost': XGBClassifier(eval_metric='logloss', random_state=42),
        'CatBoost': CatBoostClassifier(verbose=0, allow_writing_files=False, random_state=42),
        'AdaBoost': AdaBoostClassifier(random_state=42),
        'LightGBM': LGBMClassifier(verbose=-1, random_state=42),
        'Support Vector Machine': SVC(probability=True, random_state=42),
        'K-Nearest Neighbors': KNeighborsClassifier(),
        'Extra Trees': ExtraTreesClassifier(random_state=42),
        'Gradient Boosting': GradientBoostingClassifier(random_state=42),
        'Naive Bayes': GaussianNB()
    }
    model = models[name]
    if use_tuned and len(params) > 0:
        model.set_params(**params)
    return model

In [4]:
def load_split(ratio):
    folder = '../Datasets/Split_' + str(round(ratio * 100)) + '/'
    train = pd.read_csv(folder + 'Dataset_' + str(dataset_number) + '_train.csv')
    test = pd.read_csv(folder + 'Dataset_' + str(dataset_number) + '_test.csv')
    return train.drop('label', axis=1), test.drop('label', axis=1), train['label'], test['label'], str(ratio)

In [5]:
accuracy_results = []
confusion_matrix_results = []
auc_results = []
training_testing_times = []

def record(ratio, name, y_test, pred, score, training_time, testing_time):
    accuracy_results.append({'Split Ratio': ratio, 'Classifier': name, 'Accuracy': accuracy_score(y_test, pred)})
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    confusion_matrix_results.append({
        'Split Ratio': ratio, 'Classifier': name, 'True Positive': tp, 'False Negative': fn, 'True Negative': tn, 'False Positive': fp,
        'True Positive Rate': tp / (tp + fn), 'False Negative Rate': fn / (tp + fn), 'True Negative Rate': tn / (tn + fp), 'False Positive Rate': fp / (tn + fp)})
    auc_results.append({'Split Ratio': ratio, 'Classifier': name, 'AUC Value': roc_auc_score(y_test, score)})
    training_testing_times.append({'Split Ratio': ratio, 'Classifier': name, 'Training Time (s)': training_time, 'Testing Time (s)': testing_time})

for ratio in split_ratios:
    X_train, X_test, y_train, y_test, key = load_split(ratio)
    tuned_params = best_params.get(key, {}).get('Dataset_' + str(dataset_number), {})
    fitted = {}
    probas = {}
    preds = {}
    train_times = {}

    for clf_name in base_names:
        clf = make_model(clf_name, tuned_params.get(clf_name, {}))
        start_time = time.time()
        clf.fit(X_train, y_train)
        training_time = time.time() - start_time
        start_time = time.time()
        pred = clf.predict(X_test)
        proba = clf.predict_proba(X_test)[:, 1]
        testing_time = time.time() - start_time
        fitted[clf_name] = clf
        probas[clf_name] = proba
        preds[clf_name] = pred
        train_times[clf_name] = training_time
        record(ratio, clf_name, y_test, pred, proba, training_time, testing_time)

    info = top_info.get(key, {}).get('global' if top_mode == 'global' else 'Dataset_' + str(dataset_number), {})
    for top in [3, 5, 7]:
        selected = info.get(str(top), base_names[:top])
        scores = info.get('scores', {})
        weights = np.array([scores.get(n, 1.0) for n in selected])

        stack = StackingClassifier(estimators=[(n, make_model(n, tuned_params.get(n, {}))) for n in selected], final_estimator=LogisticRegression(max_iter=3000), cv=5, n_jobs=-1)
        start_time = time.time()
        stack.fit(X_train, y_train)
        training_time = time.time() - start_time
        start_time = time.time()
        pred = stack.predict(X_test)
        proba = stack.predict_proba(X_test)[:, 1]
        testing_time = time.time() - start_time
        record(ratio, 'Stacking Top ' + str(top), y_test, pred, proba, training_time, testing_time)

        start_time = time.time()
        prob_matrix = np.column_stack([probas[n] for n in selected])
        pred_matrix = np.column_stack([preds[n] for n in selected])
        votes = {
            'Hard Voting': pred_matrix.mean(axis=1),
            'Soft Voting': prob_matrix.mean(axis=1),
            'Weighted Voting': np.average(prob_matrix, axis=1, weights=weights)
        }
        vote_time = time.time() - start_time
        base_train = sum(train_times[n] for n in selected)
        for vote_name, score in votes.items():
            record(ratio, vote_name + ' Top ' + str(top), y_test, (score >= 0.5).astype(int), score, base_train, vote_time)

accuracy_df = pd.DataFrame(accuracy_results)
confusion_matrix_df = pd.DataFrame(confusion_matrix_results)
auc_df = pd.DataFrame(auc_results)
training_testing_times_df = pd.DataFrame(training_testing_times)

In [6]:
print("Accuracy:")
accuracy_df

Accuracy:


,Split Ratio,Classifier,Accuracy
0,0.7,Random Forest,0.728155
1,0.7,Logistic Regression,0.718447
2,0.7,Decision Tree,0.699029
3,0.7,XGBoost,0.728155
4,0.7,CatBoost,0.679612
...,...,...,...
67,0.9,Weighted Voting Top 5,0.685714
68,0.9,Stacking Top 7,0.714286
69,0.9,Hard Voting Top 7,0.628571
70,0.9,Soft Voting Top 7,0.742857


In [7]:
print("\nConfusion Matrix Metrics:")
confusion_matrix_df


Confusion Matrix Metrics:


,Split Ratio,Classifier,True Positive,False Negative,True Negative,False Positive,True Positive Rate,False Negative Rate,True Negative Rate,False Positive Rate
0,0.7,Random Forest,24,14,51,14,0.631579,0.368421,0.784615,0.215385
1,0.7,Logistic Regression,21,17,53,12,0.552632,0.447368,0.815385,0.184615
2,0.7,Decision Tree,21,17,51,14,0.552632,0.447368,0.784615,0.215385
3,0.7,XGBoost,24,14,51,14,0.631579,0.368421,0.784615,0.215385
4,0.7,CatBoost,29,9,41,24,0.763158,0.236842,0.630769,0.369231
...,...,...,...,...,...,...,...,...,...,...
67,0.9,Weighted Voting Top 5,9,4,15,7,0.692308,0.307692,0.681818,0.318182
68,0.9,Stacking Top 7,9,4,16,6,0.692308,0.307692,0.727273,0.272727
69,0.9,Hard Voting Top 7,9,4,13,9,0.692308,0.307692,0.590909,0.409091
70,0.9,Soft Voting Top 7,10,3,16,6,0.769231,0.230769,0.727273,0.272727


In [8]:
print("\nAUC Values:")
auc_df


AUC Values:


,Split Ratio,Classifier,AUC Value
0,0.7,Random Forest,0.789271
1,0.7,Logistic Regression,0.767409
2,0.7,Decision Tree,0.735223
3,0.7,XGBoost,0.786032
4,0.7,CatBoost,0.785830
...,...,...,...
67,0.9,Weighted Voting Top 5,0.805944
68,0.9,Stacking Top 7,0.812937
69,0.9,Hard Voting Top 7,0.814685
70,0.9,Soft Voting Top 7,0.816434


In [10]:
os.makedirs(res_dir, exist_ok=True)
merged = accuracy_df.merge(auc_df, on=['Split Ratio', 'Classifier']).merge(confusion_matrix_df, on=['Split Ratio', 'Classifier']).merge(training_testing_times_df, on=['Split Ratio', 'Classifier'])
merged.to_csv(res_dir + 'Split_Dataset_' + str(dataset_number) + '.csv', index=False)
merged.sort_values('AUC Value', ascending=False).head(10)

,Split Ratio,Classifier,Accuracy,AUC Value,True Positive,False Negative,True Negative,False Positive,True Positive Rate,False Negative Rate,True Negative Rate,False Positive Rate,Training Time (s),Testing Time (s)
54,0.9,LightGBM,0.742857,0.851399,11,2,15,7,0.846154,0.153846,0.681818,0.318182,0.027309,0.002307
33,0.8,Extra Trees,0.797101,0.836762,21,5,34,9,0.807692,0.192308,0.790698,0.209302,0.137873,0.010397
51,0.9,XGBoost,0.742857,0.830420,10,3,16,6,0.769231,0.230769,0.727273,0.272727,0.061025,0.003811
70,0.9,Soft Voting Top 7,0.742857,0.816434,10,3,16,6,0.769231,0.230769,0.727273,0.272727,1.040198,0.000000
71,0.9,Weighted Voting Top 7,0.742857,0.816434,10,3,16,6,0.769231,0.230769,0.727273,0.272727,1.040198,0.000000
69,0.9,Hard Voting Top 7,0.628571,0.814685,9,4,13,9,0.692308,0.307692,0.590909,0.409091,1.040198,0.000000
53,0.9,AdaBoost,0.714286,0.812937,9,4,16,6,0.692308,0.307692,0.727273,0.272727,0.024331,0.018865
68,0.9,Stacking Top 7,0.714286,0.812937,9,4,16,6,0.692308,0.307692,0.727273,0.272727,2.404094,0.040134
58,0.9,Gradient Boosting,0.742857,0.809441,10,3,16,6,0.769231,0.230769,0.727273,0.272727,0.048756,0.000000
66,0.9,Soft Voting Top 5,0.685714,0.805944,9,4,15,7,0.692308,0.307692,0.681818,0.318182,0.988558,0.000000
